# Weekly Log Check

**CSI 4170/5170 — Machine Learning, Fall 2026 · Oakland University**  
Jing Yee · Christina Carvalho · Pablo Avila

---

Run this **once a week** on a fresh export of the daily log sheet.

It does two jobs:

1. **Cleans** the export — fixes name and date inconsistencies, drops blank and
   incomplete rows, anonymises to P1/P2/P3, and reports every change.
2. **Checks coverage** — shows who has logged on which days, so a person who
   stops gets noticed within days instead of at the end of the semester.

The second job is the reason for the weekly rhythm. Missed days cannot be
recovered later, so finding a gap early is the only time it is fixable.

> **Before the first run:** `clean_logs.py` must be committed to `code/` in the
> repository. Cell 1 downloads it from there.

## 1 · Get the cleaning script

In [ ]:
import urllib.request, urllib.error

URL = ("https://raw.githubusercontent.com/Takilakm/"
       "ml-data-project/main/code/clean_logs.py")

try:
    urllib.request.urlretrieve(URL, "clean_logs.py")
    print("downloaded clean_logs.py")
except urllib.error.HTTPError as e:
    if e.code == 404:
        print("NOT FOUND in the repository.")
        print("Commit clean_logs.py into the code/ folder first,")
        print("or upload it with the file picker in the next cell.")
    else:
        raise

## 2 · Upload this week's export

In the sheet: **File → Download → Comma-separated values**, making sure the
Daily Log tab is the one selected. Then run this cell and pick that file.

(If cell 1 said NOT FOUND, upload `clean_logs.py` here at the same time.)

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

csvs = [f for f in uploaded if f.lower().endswith(".csv")]
if not csvs:
    raise SystemExit("No .csv uploaded — export the sheet as CSV and try again.")

RAW = csvs[0]
print("\nusing:", RAW)

## 3 · Clean it

Every change is printed. Nothing is edited by hand, so the whole process is
reproducible — which is what makes it describable in the report.

In [ ]:
import importlib, clean_logs
importlib.reload(clean_logs)

df = clean_logs.main(RAW)

## 4 · Read the coverage grid

This is the part that matters each week.

- A column of `1`s means that person is logging.
- A run of `0`s at the bottom means they have stopped — **message them today.**

The script flags anyone whose last entry is 3 or more days behind the newest.

In [ ]:
# A compact view of the last two weeks.
recent = df[df["Date"] >= df["Date"].max() - __import__("pandas").Timedelta(days=13)]
grid = (recent.assign(v=1)
              .pivot_table(index="Date", columns="Person", values="v", aggfunc="size")
              .fillna(0).astype(int))
grid.index = grid.index.strftime("%a %m-%d")
print(grid.to_string())
print("\nrows per person (all time):")
print(df["Person"].value_counts().sort_index().to_string())

## 5 · Project the final dataset size

How many rows this reaches by the Phase 2 deadline if the current rate holds.
Useful for deciding whether to change anything about collection.

In [ ]:
import pandas as pd

DEADLINE = pd.Timestamp("2026-11-13")

span = (df["Date"].max() - df["Date"].min()).days + 1
rate = len(df) / span
days_left = (DEADLINE - df["Date"].max()).days

print(f"rows so far        : {len(df)}")
print(f"days collected     : {span}")
print(f"current rate       : {rate:.1f} rows/day  (3.0 = everyone logging)")
print(f"days to Phase 2    : {days_left}")
print(f"projected total    : {len(df) + rate * days_left:.0f} rows")
print(f"if all three log   : {len(df) + 3 * days_left:.0f} rows")

## 6 · Download the results

Keep these on your computer. **Do not commit the CSV to the repository until
coverage is balanced across all three people** — a dataset that is mostly one
person is not what the study promised, and it is published under a licence with
our names on it.

In [ ]:
for f in ["daily_log_anonymised.csv", "cleaning_report.txt"]:
    if os.path.exists(f):
        files.download(f)
        print("downloaded", f)

## What to do with what you just saw

| If the grid shows… | Then |
|---|---|
| everyone logging | nothing — carry on |
| someone 3+ days behind | message them today, not at the weekend |
| a day everyone missed | leave it blank; **never invent a row** |
| the projection below ~100 rows | raise it with the group now, while it is fixable |

Missed days are left blank on purpose. The syllabus forbids submitting data not
based on your own work, and a gap in the record is a normal, reportable property
of self-collected data — it is part of what the study is about.